### 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import utils_z
import cityjson_parser_lod2 as cjpar


In [5]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [179]:
city_name = "rotterdam"
city_prefix  = "NL_RO"

### 1 处理lod2的CityGML数据

#### 1.1 测试：任选一个lod2的xml并转换为json，然后解析数据做检查

In [ ]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>




CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" --version', returncode=0, stdout='citygml-tools version 2.4.1\n(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>\n\n', stderr='')

In [ ]:
test_xml_path = f"E:\\2_data\\building_3d_opensource\\{city_name}\\lod2_gml\\DA4_3D_Buildings_Merged.gml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')

[14:52:54 INFO] Starting citygml-tools.
[14:52:54 INFO] Executing 'to-cityjson' command.
[14:52:55 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\newyork\lod2_gml\DA4_3D_Buildings_Merged.gml.
[14:52:57 WARN] The input file uses unsupported non-CityGML namespaces: http://www.w3.org/2001/SMIL20/Language, http://www.w3.org/2001/SMIL20/, http://www.ascc.net/xml/schematron.
[14:52:57 INFO] Non-CityGML content is skipped unless a matching ADE extension has been loaded.
[14:52:57 INFO] Writing output to file E:\2_data\building_3d_opensource\newyork\lod2_gml\DA4_3D_Buildings_Merged.json.
[14:53:01 INFO] Total execution time: 06 s.
[14:53:01 INFO] citygml-tools finished with 1 warning(s).



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\newyork\\lod2_gml\\DA4_3D_Buildings_Merged.gml', returncode=0, stdout="[14:52:54 INFO] Starting citygml-tools.\n[14:52:54 INFO] Executing 'to-cityjson' command.\n[14:52:55 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\newyork\\lod2_gml\\DA4_3D_Buildings_Merged.gml.\n[14:52:57 WARN] The input file uses unsupported non-CityGML namespaces: http://www.w3.org/2001/SMIL20/Language, http://www.w3.org/2001/SMIL20/, http://www.ascc.net/xml/schematron.\n[14:52:57 INFO] Non-CityGML content is skipped unless a matching ADE extension has been loaded.\n[14:52:57 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\newyork\\lod2_gml\\DA4_3D_Buildings_Merged.json.\n[14:53:01 INFO] Total execution time: 06 s.\n[14:53:01 INFO] citygml-tools finished with 1 warning(s).\n", stderr='')

In [ ]:
import json

# test_json_path = test_xml_path.replace('.xml', '.json')
# test_json_path = test_xml_path.replace('.gml', '.json')
test_json_path = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json\8-296-496.city.json"

In [ ]:
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 第一段：顶层结构和对象类型统计
print("顶层keys:", data.keys())

object_counts = {}
for obj_id, obj in data["CityObjects"].items():
    obj_type = obj["type"]
    object_counts[obj_type] = object_counts.get(obj_type, 0) + 1
print("\nCityObject类型统计:")
for k, v in sorted(object_counts.items()):
    print(f"  {k}: {v}")

顶层keys: dict_keys(['CityObjects', 'metadata', 'transform', 'type', 'version', 'vertices'])

CityObject类型统计:
  Building: 575
  BuildingPart: 577


In [ ]:
# 第二段：坐标系和transform
print("\nmetadata:", data.get("metadata", {}))
print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])


metadata: {'fullMetadataUrl': 'https://data.3dbag.nl/metadata/v20250903/metadata.json', 'geographicalExtent': [87521.5, 430877.75, -2.474000930786133, 89604.84375, 432875.6875, 45.26900100708008], 'pointOfContact': {'contactName': '3DBAG Team', 'emailAddress': 'info@3dbag.nl', 'role': 'owner', 'website': 'https://3dbag.nl'}, 'referenceSystem': 'https://www.opengis.net/def/crs/EPSG/0/7415', 'title': '3DBAG', 'version': 'v2025.09.03'}
transform: {'scale': [0.001, 0.001, 0.001], 'translate': [88563.171875, 431876.71875, 21.398000717163086]}
第一个顶点原始值: [-752850, -984399, -21398]


In [ ]:
# 第三段：第一个Building/BuildingPart的属性和几何

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        print(f"\n对象ID: {obj_id}")
        print(f"类型: {obj['type']}")
        print(f"属性: {json.dumps(obj.get('attributes', {}), indent=2, ensure_ascii=False)}")
        for geom in obj.get("geometry", []):
            print(f"几何LOD: {geom.get('lod')}, type: {geom.get('type')}, "
                  f"semantics surfaces数量: {len(geom.get('semantics', {}).get('surfaces', []))}")
        break



对象ID: NL.IMBAG.Pand.0599100000044400
类型: Building
属性: {
  "b3_bag_bag_overlap": 0.0,
  "b3_bouwlagen": null,
  "b3_dak_type": "slanted",
  "b3_extrusie": "standard",
  "b3_h_dak_50p": 11.727999687194824,
  "b3_h_dak_70p": 11.755000114440918,
  "b3_h_dak_max": 12.197999954223633,
  "b3_h_dak_min": 11.63599967956543,
  "b3_h_maaiveld": 3.6670000553131104,
  "b3_h_nok": null,
  "b3_is_glas_dak": false,
  "b3_kas_warenhuis": false,
  "b3_kwaliteitsindicator": true,
  "b3_mutatie_ahn3_ahn4": false,
  "b3_mutatie_ahn4_ahn5": false,
  "b3_n_nok": 0,
  "b3_n_vlakken": 7,
  "b3_nodata_fractie_ahn3": 0.09157554805278778,
  "b3_nodata_fractie_ahn4": 0.0,
  "b3_nodata_fractie_ahn5": 0.0007455268641933799,
  "b3_nodata_radius_ahn3": 0.0,
  "b3_nodata_radius_ahn4": 0.006519424729049206,
  "b3_nodata_radius_ahn5": 0.0,
  "b3_opp_buitenmuur": 1314.43,
  "b3_opp_dak_plat": 2127.08,
  "b3_opp_dak_schuin": 0.0,
  "b3_opp_grond": 2012.21,
  "b3_opp_scheidingsmuur": 276.28,
  "b3_puntdichtheid_ahn3": 7.07

In [ ]:
# 看所有surface类型
surface_types = set()
for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        if str(geom.get("lod")) == "2":
            for s in geom.get("semantics", {}).get("surfaces", []):
                surface_types.add(s.get("type"))
print("surface类型:", surface_types)

# 看LOD分布
lod_counts = {}
for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        lod = str(geom.get("lod"))
        lod_counts[lod] = lod_counts.get(lod, 0) + 1
print("LOD分布:", lod_counts)

surface类型: set()
LOD分布: {'0': 575, '1.2': 577, '1.3': 577, '2.2': 577}


In [ ]:
test_buildings = cjpar.parse_cityjson_lod2(test_json_path, "2")
print(f"解析出建筑数：{len(test_buildings)}")
b = test_buildings[0] # 第一个
print(f"citygml_id: {b['citygml_id']}")
print(f"floor_count: {b['floor_count']}")
print(f"function: {b['function']}")
print(f"roof_type: {b['roof_type']}")
print(f"year_built: {b['year_built']}")
print(f"高度：{b['height']}")
print(f"2D footprint：{b['geom_2d']}")
print(f"RoofSurface数量：{len(b['surfaces']['RoofSurface'])}")
print(f"WallSurface数量：{len(b['surfaces']['WallSurface'])}")
print(f"GroundSurface数量：{len(b['surfaces']['GroundSurface'])}")

解析出建筑数：16665
citygml_id: gml_J9693EVO70UM2Z75C9RXI0UQUKNOCC8BVY1P
floor_count: None
function: None
roof_type: None
year_built: None
高度：26.174
2D footprint：POLYGON ((1025257 184445.66700000002, 1025258.4369999999 184438.513, 1025266.4759999999 184440.127, 1025268.575 184429.676, 1025263.7379999999 184428.70500000002, 1025265.382 184420.52, 1025220.1089999999 184411.428, 1025214.929 184437.218, 1025257 184445.66700000002))
RoofSurface数量：1
WallSurface数量：8
GroundSurface数量：1


##### 检查数据规范性

In [ ]:
# 试几个捷克常用坐标系
for srid in [5514, 2065, 32633]:
    try:
        result = utils_z.run_sql(f"""
            SELECT ST_AsText(ST_Transform(
                ST_SetSRID(ST_MakePoint(-732503, -1053555), {srid}), 4326));
        """, conn=conn, fetch=True)
        print(f"EPSG:{srid} -> {result[0][0]}")
    except Exception as e:
        print(f"EPSG:{srid} -> 错误: {e}")

EPSG:5514 -> POINT(14.583748244966255 50.006083245735894)
EPSG:2065 -> POINT(46.64294205115266 65.12139873321648)
EPSG:32633 -> POINT(3.843385550549216 -9.352889711007553)


In [ ]:
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "Building":
        for geom in obj.get("geometry", []):
            if str(geom.get("lod")) == "3":
                print("semantics:", json.dumps(geom.get("semantics", {}), indent=2)[:300])
                break
        break

semantics: {
  "surfaces": [
    {
      "type": "RoofSurface",
      "Area": 123.557354,
      "Steepness": 42.762,
      "Exposition": -46.135,
      "ID_intern": "BD3_Prah26_1_47"
    },
    {
      "type": "RoofSurface",
      "Area": 43.830175,
      "Steepness": 51.924,
      "Exposition": 43.081,
      


In [ ]:
import numpy as np

with open(r"E:\2_data\building_3d_opensource\prague\lod2_gml\Prah_2-6.json", "r", encoding="utf-8") as f:
    data = json.load(f)

scale         = np.array(data["transform"]["scale"])
translate     = np.array(data["transform"]["translate"])
vertices      = np.array(data["vertices"])
real_vertices = vertices * scale + translate

x, y = real_vertices[0][0], real_vertices[0][1]
print(f"原始坐标: x={x:.3f}, y={y:.3f}")

r1 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint({x}, {y}), 5514), 4326));
""", conn=conn, fetch=True)

r2 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint({y}, {x}), 5514), 4326));
""", conn=conn, fetch=True)

print(f"X,Y顺序: {r1[0][0]}")
print(f"Y,X顺序: {r2[0][0]}")

原始坐标: x=-732436.256, y=-1052006.570
X,Y顺序: POINT(14.581771910426369 50.01996021768607)
Y,X顺序: POINT(9.300398487853869 52.3663701942263)


In [ ]:
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "BuildingPart":
        print(f"BuildingPart属性: {obj.get('attributes', {})}")
        break

BuildingPart属性: {'usage': 'vodorovna stresni plocha'}


In [ ]:
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "Building":
        print(f"Building属性: {json.dumps(obj.get('attributes', {}), indent=2, ensure_ascii=False)}")
        # 同时看children关系
        print(f"children: {obj.get('children', [])}")
        break

Building属性: {
  "citygrid_UnitID": "202083",
  "usage": "sikma stresni plocha"
}
children: ['ID_c5e7cf3b-1aaf-405f-9f12-49fb3ef1690c']


In [ ]:
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "BuildingPart":
        for geom in obj.get("geometry", []):
            if str(geom.get("lod")) == "3":
                semantics = geom.get("semantics", {})
                print(f"surfaces数量: {len(semantics.get('surfaces', []))}")
                print(f"values类型: {type(semantics.get('values', []))}")
                print(f"values示例: {str(semantics.get('values', []))[:100]}")
                print(f"surface types: {[s.get('type') for s in semantics.get('surfaces', [])]}")
                break
        break

surfaces数量: 5
values类型: <class 'list'>
values示例: [[0, 1, 2, 3, 4]]
surface types: ['RoofSurface', 'WallSurface', 'WallSurface', 'WallSurface', 'GroundSurface']


#### 1.2 变量

In [ ]:
block_table_name = f"blocks.{city_name}_blocks"
lod2_table_name_full = f"lod2.{city_name}_buildings_lod2" #TODO: 加前缀是否有问题
lod2_table_name = f"{city_name}_buildings_lod2"
lod2_surface_table_name_full  = f"lod2.{city_name}_building_surfaces_lod2"
lod2_surface_table_name  = f"{city_name}_building_surfaces_lod2"

target_srid  = 4326
source_srid  = 28992

#### 1.3 创建LOD2建筑表并批量转换、导入数据

##### 1.3.1 批量转换xml/gml -> json (如需)

In [165]:
# 先批量转换XML到CityJSON
lod2_xml_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_gml"
lod2_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json"
os.makedirs(lod2_json_dir, exist_ok=True)

In [166]:
# xml_files = [f for f in os.listdir(lod2_xml_dir) if f.endswith(".xml")]
xml_files = [f for f in os.listdir(lod2_xml_dir) if f.endswith(".gml")]
print(f"共{len(xml_files)}个文件待转换")

errors = [] 
for i, filename in enumerate(xml_files):
    input_path = os.path.join(lod2_xml_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{lod2_json_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd, False)
        if (i + 1) % 50 == 0:
            print(f"转换进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"转换错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

共20个文件待转换
转换完成，失败0个


##### 1.3.2 建表入库

In [192]:
lod2_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json"

In [ ]:
# LOD2建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod2_table_name_full} (
        building_id     VARCHAR PRIMARY KEY 
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        roof_type       VARCHAR,
        year_built      INTEGER
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_table_name}_geom_idx
    ON {lod2_table_name_full} USING GIST (geom_2d);
""", conn=conn)

# LOD2 surface子表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod2_surface_table_name_full} (
        surface_id      VARCHAR PRIMARY KEY
        citygml_id      VARCHAR,
        building_id     VARCHAR REFERENCES {lod2_table_name_full}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_surface_table_name}_building_idx
    ON {lod2_surface_table_name_full} (building_id);
""", conn=conn)

print(city_prefix+" LOD2表创建完成")

NL_RO LOD2表创建完成


In [188]:
import traceback

In [ ]:
json_files = [f for f in os.listdir(lod2_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

total = 0
errors = []

conn.rollback()
# 获取当前最大ID，设置计数器初始值（空表格则为1）
cur = conn.cursor()

cur.execute(f"SELECT MAX(building_id) FROM {lod2_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1

cur.execute(f"SELECT MAX(surface_id) FROM {lod2_surface_table_name_full}")
max_sid = cur.fetchone()[0]
surface_counter = int(max_sid.split("_S_")[1]) + 1 if max_sid else 1

cur.close()

# 遍历json文件，解析并入库，如有一个出错，整个文件跳过
for i, filename in enumerate(json_files):
    filepath = os.path.join(lod2_json_dir, filename)
    try:
        # buildings = cjpar.parse_cityjson_lod2(filepath, target_lod="2")
        buildings = cjpar.parse_cityjson_lod2_NL_AM(filepath)
        count, building_counter, surface_counter = cjpar.insert_buildings_lod2(
            buildings, conn,
            lod2_table=lod2_table_name_full,
            surface_table=lod2_surface_table_name_full,
            city_prefix=city_prefix,
            target_srid=target_srid,
            source_srid=source_srid,
            building_counter=building_counter,
            surface_counter=surface_counter
        )
        total += count
        if (i + 1) % 50 == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}，已入库表面：{surface_counter-1}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()

print(f"\n完成！共入库建筑：{total}，共入库表面：{surface_counter-1}")

print(f"失败文件数：{len(errors)}")

共125个文件待入库
入库进度：50/125，已入库建筑：73623，已入库表面：1071528
入库进度：100/125，已入库建筑：142585，已入库表面：2256535

完成！共入库建筑：189649，共入库表面：2910029
失败文件数：0


#### 1.3 更新block_id字段并空间叠合

In [ ]:
# 确认空间索引
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_table_name}_geom_idx
    ON {lod2_table_name_full} USING GIST (geom_2d);
""", conn=conn)
print("开始空间叠合...")

开始空间叠合...


In [ ]:
utils_z.run_sql(f"""
    UPDATE {lod2_table_name_full} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")


空间叠合完成


In [ ]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod2_table_name_full};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：189649
成功匹配block：154002
未匹配block：35647


In [ ]:
# 检查包含LOD2建筑的街区数量
sql_counts = f"SELECT COUNT(DISTINCT block_id) FROM {lod2_table_name_full} WHERE block_id IS NOT NULL;"
(lod2_count,) = utils_z.run_sql(sql_counts, conn=conn, fetch=True)[0]

print(f"包含 LOD2 建筑的街区数量: {lod2_count}")

包含 LOD2 建筑的街区数量: 2793


##### debug Namur

In [ ]:
print(utils_z.run_sql(f"SELECT ST_Extent(geom_2d) FROM {lod2_table_name_full};", conn=conn, fetch=True))
print(utils_z.run_sql(f"SELECT ST_Extent(geom) FROM blocks.namur_blocks;", conn=conn, fetch=True))

[('BOX(4.715207859164983 50.389523113851766,4.973222426952733 50.53155855525857)',)]
[('BOX(4.725290985752069 50.38973709936696,4.9748267 50.52564369933891)',)]


In [68]:
result = utils_z.run_sql("""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint(184452, 127995), 31300), 4326));
""", conn=conn, fetch=True)
print("31300转4326测试点:", result[0][0])

31300转4326测试点: POINT(4.843425319978331 50.46177483672698)


In [ ]:
# 取一栋建筑，看它的所有surface的Z值分布
result = utils_z.run_sql(f"""
    SELECT surface_type,
           ST_ZMin(geom_3d) as z_min,
           ST_ZMax(geom_3d) as z_max
    FROM {lod2_surface_table_name_full}
    WHERE building_id = (
        SELECT building_id FROM {lod2_table_name_full} 
        WHERE block_id IS NOT NULL LIMIT 1
    )
    ORDER BY surface_type, z_min;
""", conn=conn, fetch=True)

for row in result:
    print(f"type={row[0]}, z_min={row[1]:.2f}, z_max={row[2]:.2f}")

type=GroundSurface, z_min=195.33, z_max=195.33
type=RoofSurface, z_min=198.50, z_max=199.21
type=WallSurface, z_min=195.33, z_max=199.21
type=WallSurface, z_min=195.33, z_max=199.21
type=WallSurface, z_min=195.33, z_max=198.50
type=WallSurface, z_min=195.33, z_max=199.21


In [71]:
import numpy as np

# 取第一个顶点的原始坐标
with open(r"E:\2_data\building_3d_opensource\namur\lod2_json\maille-50.json", "r", encoding="utf-8") as f:
    data = json.load(f)

scale         = np.array(data["transform"]["scale"])
translate     = np.array(data["transform"]["translate"])
vertices      = np.array(data["vertices"])
real_vertices = vertices * scale + translate

x, y = real_vertices[0][0], real_vertices[0][1]
print(f"原始坐标: x={x:.3f}, y={y:.3f}")

# 试两种轴顺序
r1 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint({x}, {y}), 31300), 4326));
""", conn=conn, fetch=True)

r2 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint({y}, {x}), 31300), 4326));
""", conn=conn, fetch=True)

print(f"X,Y顺序: {r1[0][0]}")
print(f"Y,X顺序: {r2[0][0]}")

原始坐标: x=193205.351, y=131471.503
X,Y顺序: POINT(4.96710550065466 50.49244810039099)
Y,X顺序: POINT(4.093979543026097 51.04871563511026)


In [ ]:
# 取一栋有block_id的建筑，看它的centroid和block centroid的距离
result = utils_z.run_sql(f"""
    SELECT 
        b.building_id,
        ST_AsText(ST_Centroid(b.geom_2d)) as building_center,
        ST_AsText(ST_Centroid(bl.geom)) as block_center,
        ST_Distance(
            ST_Centroid(b.geom_2d)::geography,
            ST_Centroid(bl.geom)::geography
        ) as distance_m
    FROM {lod2_table_name_full} b
    JOIN blocks.namur_blocks bl ON b.block_id = bl.block_id
    LIMIT 5;
""", conn=conn, fetch=True)

for row in result:
    print(f"building: {row[1]}")
    print(f"block center: {row[2]}")
    print(f"距离: {row[3]:.1f}米")
    print("---")

building: POINT(4.882345371031527 50.490008118601246)
block center: POINT(4.881835851541169 50.49071599815786)
距离: 86.6米
---
building: POINT(4.882639534742761 50.49650218464328)
block center: POINT(4.881983515965765 50.49779301718394)
距离: 150.9米
---
building: POINT(4.879429753826257 50.41754864955123)
block center: POINT(4.872237543223382 50.42846877274435)
距离: 1317.9米
---
building: POINT(4.87748472051003 50.43954830898036)
block center: POINT(4.874537830963286 50.4416933642482)
距离: 317.4米
---
building: POINT(4.878721541028889 50.454654857182824)
block center: POINT(4.880625056994662 50.451674358745734)
距离: 358.0米
---


In [ ]:
# 取第一栋建筑的centroid
result = utils_z.run_sql(f"""
    SELECT building_id, ST_AsText(ST_Centroid(geom_2d))
    FROM {lod2_table_name_full}
    WHERE block_id IS NOT NULL
    LIMIT 1;
""", conn=conn, fetch=True)

building_id = result[0][0]
building_centroid = result[0][1]
print(f"建筑: {building_id}")
print(f"建筑centroid: {building_centroid}")

# 看它现在被分配到哪个block
result2 = utils_z.run_sql(f"""
    SELECT block_id FROM {lod2_table_name_full}
    WHERE building_id = '{building_id}';
""", conn=conn, fetch=True)
print(f"当前block_id: {result2[0][0]}")

# 看它的centroid实际落在哪个block里
result3 = utils_z.run_sql(f"""
    SELECT block_id, ST_AsText(geom)
    FROM blocks.namur_blocks
    WHERE ST_Within(
        ST_GeomFromText('{building_centroid}', 4326),
        geom
    );
""", conn=conn, fetch=True)
print(f"实际应该在的block: {result3}")

建筑: BE_NA_B_0010454
建筑centroid: POINT(4.878721541028889 50.454654857182824)
当前block_id: BE_NA_000572
实际应该在的block: [('BE_NA_000572', 'POLYGON((4.876956399999999 50.44993899935441,4.8769919 50.450378399354314,4.877102999999999 50.45130569935412,4.877205299999999 50.45157189935408,4.8774057 50.452117499353974,4.8775785 50.45307069935375,4.8776217 50.45329229935373,4.8777202 50.45352419935366,4.8778559 50.453733999353624,4.878404399999999 50.45422169935352,4.878620199999999 50.45450509935347,4.8786174 50.45464799935344,4.878810799999999 50.45468809935342,4.8790868 50.45478559935341,4.879546899999999 50.45500679935336,4.8800231 50.455265499353324,4.8800959 50.45530639935329,4.880175199999999 50.455351099353294,4.8802842 50.455410699353294,4.880801699999999 50.45568649935323,4.881600699999999 50.45612929935314,4.8818911 50.45625629935311,4.8820834 50.456310399353065,4.8822666 50.456330399353085,4.882487899999999 50.45632279935309,4.8826512 50.456298099353106,4.8827948 50.4562663993531,4.8831

In [ ]:
# 取一栋有block的建筑，打印它的完整geom_2d
result = utils_z.run_sql(f"""
    SELECT building_id, ST_AsText(geom_2d)
    FROM {lod2_table_name_full}
    WHERE block_id IS NOT NULL
    LIMIT 1;
""", conn=conn, fetch=True)

print(result[0][0])
print(result[0][1])

BE_NA_B_0010454
POLYGON((4.878726928129733 50.45463634407644,4.878693035290855 50.4546596824642,4.878716153929818 50.454673370291204,4.878750046767101 50.45465003189675,4.878726928129733 50.45463634407644))


In [76]:
with open(r"E:\2_data\building_3d_opensource\namur\lod2_json\maille-50.json", "r", encoding="utf-8") as f:
    data = json.load(f)

scale         = np.array(data["transform"]["scale"])
translate     = np.array(data["transform"]["translate"])
vertices      = np.array(data["vertices"])
real_vertices = vertices * scale + translate

x, y = real_vertices[0][0], real_vertices[0][1]
print(f"测试点原始坐标: x={x:.3f}, y={y:.3f}")

# 三种转换方式
r1 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint({x}, {y}), 31300), 4326));
""", conn=conn, fetch=True)
print(f"直接转换: {r1[0][0]}")

r2 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_Transform(
            ST_SetSRID(ST_MakePoint({x}, {y}), 31300),
        3812),
    4326));
""", conn=conn, fetch=True)
print(f"经3812中转: {r2[0][0]}")

r3 = utils_z.run_sql(f"""
    SELECT ST_AsText(ST_Transform(
        ST_SetSRID(ST_MakePoint({x}, {y}), 31370), 4326));
""", conn=conn, fetch=True)
print(f"用31370: {r3[0][0]}")

测试点原始坐标: x=193205.351, y=131471.503
直接转换: POINT(4.96710550065466 50.49244810039099)
经3812中转: POINT(4.96710550065466 50.49244810131665)
用31370: POINT(4.97765272817026 50.492447649270105)


In [75]:
# 用PostGIS直接检查转换
result = utils_z.run_sql("""
    SELECT ST_AsText(
        ST_Transform(
            ST_SetSRID(ST_MakePoint(75000, 75000), 2169),
            4326
        )
    );
""", conn=conn, fetch=True)
print("2169转4326测试点:", result[0][0])

2169转4326测试点: POINT(6.09894264813424 49.6096093866702)


In [ ]:
# 1. 检查两张表各自的坐标系
print(utils_z.run_sql(f"SELECT Find_SRID('public', '{lod2_table_name_full}', 'geom_2d');", conn=conn, fetch=True))
print(utils_z.run_sql(f"SELECT Find_SRID('public', '{block_table_name}', 'geom');", conn=conn, fetch=True))

# 2. 看看两张表的坐标范围是否在同一数量级
print(utils_z.run_sql(f"SELECT ST_Extent(geom_2d) FROM {lod2_table_name_full};", conn=conn, fetch=True))
print(utils_z.run_sql(f"SELECT ST_Extent(geom) FROM {block_table_name};", conn=conn, fetch=True))

# 3. 随便取一栋建筑的centroid，看它落在哪里
print(utils_z.run_sql(f"SELECT ST_AsText(ST_Centroid(geom_2d)) FROM {lod2_table_name_full} LIMIT 1;", conn=conn, fetch=True))

[(25833,)]
[(0,)]
[('BOX(-10749.27 331389.01,18183.795 353381.93799999997)',)]
[('BOX(589200.6788676552 5330441.177434857,614929.5032929091 5351551.6025656825)',)]
[('POINT(-10212.01 345059.78)',)]


In [ ]:
# 先确认block表建表时的transform目标
# 再用ST_SetSRID强制修正block表的坐标系（如果数据本身是对的只是没登记）
utils_z.run_sql(f"""
    SELECT ST_AsText(geom) FROM {block_table_name} LIMIT 1;
""", conn=conn, fetch=True)


[('POLYGON((589502.8670227258 5342245.796037353,589441.5448576884 5342292.750634647,589406.7026083237 5342314.294902007,589335.6474588497 5342348.668719315,589285.2988223999 5342363.378227262,589251.7038950417 5342371.735374545,589200.6788676552 5342384.600651726,589215.5367276989 5342441.976828558,589222.653586947 5342461.877248411,589250.7802335394 5342501.984416407,589254.2822857167 5342506.986490045,589284.864469873 5342485.096716668,589309.2792616172 5342467.624236941,589489.3078402273 5342338.758993452,589498.2153494882 5342332.539530742,589508.1986397573 5342325.458679183,589518.2171686984 5342318.500689847,589526.9101452322 5342311.755381777,589543.399448982 5342297.18345026,589537.5657849787 5342289.765458014,589502.8670227258 5342245.796037353))',)]

In [ ]:
# 同时看一眼building原始坐标是否合理
utils_z.run_sql(f"""
    SELECT ST_AsText(geom_2d) FROM {lod2_table_name_full} LIMIT 1;
""", conn=conn, fetch=True)

[('POLYGON((-10215.689999999999 345057.75,-10214.21 345063.36,-10208.33 345061.81,-10209.81 345056.2,-10215.689999999999 345057.75))',)]

In [23]:
with open("E:\\2_data\\building_3d_opensource\\vienna\\lod2_json\\079090.json", "r") as f:
    data = json.load(f)

print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])
print("metadata:", data.get("metadata", {}))

transform: {'scale': [0.001, 0.001, 0.001], 'translate': [-10284.81, 344996.82, 129.24]}
第一个顶点原始值: [277792, 442497, 22964]
metadata: {'geographicalExtent': [-10284.81, 344996.82, 129.24, -9996.12, 345465.06, 157.12], 'referenceSystem': 'http://www.opengis.net/def/crs/EPSG/0/31256'}


##### debug New York

In [199]:
ny_building_table = "newyork_buildings_lod2"

In [ ]:
utils_z.run_sql(f"""
    SELECT 
        median_height,
        median_height * 0.3048 as if_feet_to_meters
    FROM (
        SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY height) as median_height
        FROM {ny_building_table}
    ) t;
""", conn=conn, fetch=True)

### 2. 一些临时处理

#### 删除seq

In [9]:
# 城市列表
cities = [
    'amsterdam', 'dresden', 'hamburg', 'hannover', 'linz', 
    'luxembourg', 'montreal', 'namur', 'newyork', 'prague', 
    'rotterdam', 'vienna', 'zurich'
]

# 表后缀
table_suffixes = ['buildings_lod2', 'building_surfaces_lod2']

table_schema = "lod2"
seq_schema = "public"

In [12]:
for city in cities:
        for suffix in table_suffixes:
            base_name = f"{city}_{suffix}"
            
            # 显式拼接带 Schema 的路径，并加双引号防止解析错误
            full_table_path = f'"{table_schema}"."{base_name}"'
            # 序列留在 public
            full_seq_path = f'"{seq_schema}"."{base_name}_seq"'
            
            print(f"正在处理: {full_table_path}")
            
            # 1. 解绑（这里只需要 DROP DEFAULT，不需要管 seq 在哪，因为它是表的一个属性）
            drop_default_sql = f'ALTER TABLE {full_table_path} ALTER COLUMN surface_id DROP DEFAULT;'
            
            # 2. 删除旧的序列（必须指明在 public 下）
            drop_seq_sql = f'DROP SEQUENCE IF EXISTS {full_seq_path};'
            
            try:
                # 先执行解绑
                utils_z.run_sql(drop_default_sql, conn=conn)
                # 再删除序列
                utils_z.run_sql(drop_seq_sql, conn=conn)
                print(f"  ✅ 成功清理 {base_name}")
            except Exception as e:
                print(f"  ❌ 出错 {base_name}: {e}")

正在处理: "lod2"."amsterdam_buildings_lod2"
  ❌ 出错 amsterdam_buildings_lod2: 错误:  关系 "amsterdam_buildings_lod2" 的 "surface_id" 字段不存在

正在处理: "lod2"."amsterdam_building_surfaces_lod2"
  ✅ 成功清理 amsterdam_building_surfaces_lod2
正在处理: "lod2"."dresden_buildings_lod2"
  ❌ 出错 dresden_buildings_lod2: 错误:  关系 "dresden_buildings_lod2" 的 "surface_id" 字段不存在

正在处理: "lod2"."dresden_building_surfaces_lod2"
  ✅ 成功清理 dresden_building_surfaces_lod2
正在处理: "lod2"."hamburg_buildings_lod2"
  ❌ 出错 hamburg_buildings_lod2: 错误:  关系 "hamburg_buildings_lod2" 的 "surface_id" 字段不存在

正在处理: "lod2"."hamburg_building_surfaces_lod2"
  ✅ 成功清理 hamburg_building_surfaces_lod2
正在处理: "lod2"."hannover_buildings_lod2"
  ❌ 出错 hannover_buildings_lod2: 错误:  关系 "hannover_buildings_lod2" 的 "surface_id" 字段不存在

正在处理: "lod2"."hannover_building_surfaces_lod2"
  ✅ 成功清理 hannover_building_surfaces_lod2
正在处理: "lod2"."linz_buildings_lod2"
  ❌ 出错 linz_buildings_lod2: 错误:  关系 "linz_buildings_lod2" 的 "surface_id" 字段不存在

正在处理: "lod2"."linz_building_su